## 1. 設定

エージェントへの指示と実行条件。

In [ ]:
SYSTEM_PROMPT = """あなたはデータサイエンスのコンペティションに参加しています。
ツールを使ってデータを確認し、モデルを作って予測を提出してください。"""

UOB_ADDRESS = "https://ncupbl.f5.si/"  # 末尾の / まで書く
MCP_SERVER_SRC = "/content/drive/MyDrive/mcp_server"  # 配布された mcp_server フォルダ
COMPETITION_KEYWORD = "2026"  # 参加するコンペの名前に含まれる文字列
MODEL = "google/gemini-3.1-flash-lite"  # OpenRouterのモデルID
NUM_ROUNDS = 3  # 提出する回数
MAX_TOOL_CALLS = 15  # 1ラウンドでツールを呼べる回数の上限
RUN_PYTHON_TIMEOUT_SEC = 300  # run_python 1回の時間制限（秒）
ALLOW_DISCUSSION_POST = False  # Trueでディスカッションへの投稿を許可

## 2. 環境構築

ドライブの接続、ライブラリと MCP サーバーの準備。

In [ ]:
import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

WORK = Path("/content/uob_agent")  # ランタイムを終了すると消える
CODE_DIR = WORK / "agent_workspace"  # run_python の作業ディレクトリ
SUBMISSION_FILE = WORK / "submissions" / "prediction.csv"
WORK.mkdir(parents=True, exist_ok=True)

# Colab の stderr は fileno() を持たず MCP の起動に失敗するため、ファイルに差し替える
sys.stderr = open(WORK / "mcp_stderr.log", "w")
print("作業フォルダ:", WORK)

In [ ]:
%pip install -q "fastmcp" "mcp>=2,<3" openai pyyaml gdown
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version

In [ ]:
import shutil

import yaml

src = Path(MCP_SERVER_SRC)
if not (src / "server_stdio.py").exists():
    raise FileNotFoundError(
        f"{src} に server_stdio.py がありません。mcp_server フォルダの場所（MCP_SERVER_SRC）を確認してください"
    )

MCP_DIR = WORK / "mcp_server"
TOKEN_PATH = WORK / "token.json"
DATA_DIR = WORK / "data"

if MCP_DIR.exists():
    shutil.rmtree(MCP_DIR)
shutil.copytree(src, MCP_DIR, ignore=shutil.ignore_patterns(".venv", "__pycache__", "*.json"))

config_path = MCP_DIR / "mcp_config.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config["token_path"] = str(TOKEN_PATH)
config["uob_adress"] = UOB_ADDRESS
config["data_dir"] = str(DATA_DIR)
config_path.write_text(yaml.safe_dump(config, allow_unicode=True), encoding="utf-8")

for d in (DATA_DIR, CODE_DIR, SUBMISSION_FILE.parent):
    d.mkdir(parents=True, exist_ok=True)

!cd "{MCP_DIR}" && uv sync -q
print("MCPサーバーの準備完了:", MCP_DIR)

## 3. ログイン

ビッグデータ大学のユーザー名とパスワードを入力する。

In [ ]:
import importlib.util

import requests

spec = importlib.util.spec_from_file_location("token_refresh", MCP_DIR / "token_refresh.py")
token_refresh = importlib.util.module_from_spec(spec)
spec.loader.exec_module(token_refresh)

try:
    token_refresh.get_access_token(verbose=True)
except requests.HTTPError as e:
    status = e.response.status_code if e.response is not None else "?"
    if status == 401:
        print(
            "ログイン失敗（401）"
        )
    else:
        print(f"ログイン失敗（{status}）: {e}")
except requests.RequestException as e:
    print(f"サーバーに接続できませんでした。もう一度このセルを実行してください: {e}")
else:
    print("ログイン")

## 4. エージェントの準備

ツールとエージェントの処理を定義する（ここではまだ実行しない）。

In [ ]:
import asyncio
import json
import subprocess
from datetime import datetime

from google.colab import userdata
from openai import OpenAI, RateLimitError

llm = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))

ALLOWED_UOB_TOOLS = {
    "get_competitions",
    "get_competitions_dataset_info",
    "download_competition_dataset",
    "get_competitions_post",
    "get_all_competitions_post",
    "get_competitions_leaderboard",
    "get_competitions_discussion_topic",
    "get_competitions_discussion_detail",
    "post_competitions_post",
}
if ALLOW_DISCUSSION_POST:
    ALLOWED_UOB_TOOLS |= {"post_competitions_discussion_topic", "post_competitions_discussion_detail"}

RUN_PYTHON_TOOL = {
    "type": "function",
    "function": {
        "name": "run_python",
        "description": (
            "Pythonコードを新しいプロセスで実行し、標準出力と標準エラーを返します。"
            "呼び出しをまたいで残るのはファイルだけです。変数やimportは残らないため、"
            "毎回ライブラリのimportとデータの読み込みからやり直してください。"
            "途中の結果を使い回したい場合はファイルに保存してください。"
            "ダウンロードしたデータセットがzipファイルなら、展開してから読み込んでください。"
            "インターネットに接続できます。データセットの説明（get_competitions_dataset_info）に"
            "学習データのGoogleドライブのリンクがある場合は、gdownで取得してください"
            "（フォルダのリンクは gdown.download_folder(url, output='train_data', quiet=True)、"
            "ファイルのリンクは gdown.download(url, output='ファイル名', quiet=True, fuzzy=True)）。"
            "取得したファイルは作業ディレクトリに残るので、2回目以降は取り直さなくてかまいません。"
            "使えるライブラリ: pandas, numpy, scikit-learn, lightgbm, xgboost, gdown。"
            f"1回の実行は{RUN_PYTHON_TIMEOUT_SEC}秒でタイムアウトします。"
        ),
        "parameters": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "実行するPythonコード"}},
            "required": ["code"],
        },
    },
}

LOG_PATH = WORK / "events.jsonl"


def log(kind: str, text: str) -> None:
    marks = {"submission": "★", "error": "✗"}
    print(f"[{datetime.now():%H:%M:%S}] {marks.get(kind, '▶')} {kind}: {text[:500]}")
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"ts": datetime.now().isoformat(), "kind": kind, "text": text}, ensure_ascii=False) + "\n")


def run_python(code: str) -> str:
    history = CODE_DIR / "generated_history"
    history.mkdir(parents=True, exist_ok=True)
    (history / f"{datetime.now():%Y%m%dT%H%M%S%f}.py").write_text(code, encoding="utf-8")
    (CODE_DIR / "generated.py").write_text(code, encoding="utf-8")
    env = {k: v for k, v in os.environ.items() if k != "OPENROUTER_API_KEY"}
    try:
        r = subprocess.run(
            ["python", "generated.py"], cwd=CODE_DIR, env=env,
            capture_output=True, text=True, timeout=RUN_PYTHON_TIMEOUT_SEC, check=False,
        )
    except subprocess.TimeoutExpired:
        return f"TIMEOUT ({RUN_PYTHON_TIMEOUT_SEC}s)"
    return f"exit={r.returncode}\n--- stdout ---\n{r.stdout[-3000:]}\n--- stderr ---\n{r.stderr[-2000:]}"


def tool_text(result) -> str:
    return "\n".join(getattr(c, "text", str(c)) for c in result.content)


# mcp の ClientSession と fastmcp の Client のどちらでも動くようにする
async def list_tools_for_llm(session) -> list:
    listed = await session.list_tools()
    tools = []
    for t in getattr(listed, "tools", listed):
        if t.name in ALLOWED_UOB_TOOLS:
            schema = getattr(t, "input_schema", None) or getattr(t, "inputSchema", None)
            tools.append({"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": schema}})
    return tools + [RUN_PYTHON_TOOL]


async def find_competition(session) -> int:
    payload = json.loads(tool_text(await session.call_tool("get_competitions", {})))
    competitions = payload.get("results", []) if isinstance(payload, dict) else payload
    matched = [c for c in competitions if COMPETITION_KEYWORD.casefold() in str(c.get("title", "")).casefold()]
    if len(matched) != 1:
        titles = [f"{c.get('id')}: {c.get('title')}" for c in competitions]
        raise RuntimeError(
            f"名前に {COMPETITION_KEYWORD!r} を含むコンペが{len(matched)}件です。"
            f"設定セルの COMPETITION_KEYWORD を直してください。参加できるコンペ: {titles}"
        )
    log("message", f"対象コンペ: {matched[0]['id']}: {matched[0]['title']}")
    return int(matched[0]["id"])


async def ask_llm(messages: list, tools: list):
    delay = 5
    for attempt in range(4):
        try:
            return await asyncio.to_thread(
                llm.chat.completions.create, model=MODEL, messages=messages,
                tools=tools, tool_choice="auto", temperature=0.2,
            )
        except RateLimitError:
            if attempt == 3:
                raise
            log("message", f"LLMが混雑しています。{delay}秒待って再試行します")
            await asyncio.sleep(delay)
            delay *= 2


async def run_round(session, competition_id: int, round_num: int, last_summary: str | None) -> str | None:
    tools = await list_tools_for_llm(session)
    task = (
        f"ラウンド{round_num}: コンペ{competition_id}に提出してください。"
        f"予測を次のパスに保存し、同じパスをupload_fileとしてpost_competitions_postを呼んでください: {SUBMISSION_FILE}。"
        "提出ファイルの形式: ヘッダー行なし、予測値1列のみ、予測対象データと同じ行数"
        "（データの説明に「1行」とあっても、1列で行数を合わせる）。"
    )
    if last_summary:
        task += f"\n\n前回のラウンドの最後の発言: {last_summary[:500]}"
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]

    msg = None
    for _ in range(MAX_TOOL_CALLS):
        msg = (await ask_llm(messages, tools)).choices[0].message
        if msg.content:
            log("model", msg.content)
        if not msg.tool_calls:
            break
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
                for c in msg.tool_calls
            ],
        })
        for call in msg.tool_calls:
            name = call.function.name
            try:
                args = json.loads(call.function.arguments or "{}")
            except json.JSONDecodeError as e:
                text = f"引数のJSONを読めませんでした: {e}"
            else:
                log("tool_call", f"{name} {json.dumps(args, ensure_ascii=False)[:300]}")
                if name == "run_python":
                    text = await asyncio.to_thread(run_python, args.get("code", ""))
                elif name in ALLOWED_UOB_TOOLS:
                    try:
                        result = await session.call_tool(name, args)
                    except Exception as e:
                        text = f"{type(e).__name__}: {e}"
                    else:
                        text = tool_text(result)
                        if name == "post_competitions_post" and not getattr(result, "is_error", False):
                            log("submission", text)
                else:
                    text = "このツールは許可されていません"
            log("tool_result", f"{name}: {text}")
            messages.append({"role": "tool", "tool_call_id": call.id, "name": name, "content": text})
    return msg.content if msg is not None else None


async def run_agent(session) -> None:
    competition_id = await find_competition(session)
    summary = None
    for round_num in range(1, NUM_ROUNDS + 1):
        log("message", f"===== ラウンド{round_num}/{NUM_ROUNDS} =====")
        summary = await run_round(session, competition_id, round_num, summary)
    log("message", "すべてのラウンドが終わりました")


print("準備完了。モデル:", MODEL, "/ 使えるツール:", sorted(ALLOWED_UOB_TOOLS | {"run_python"}))

## 5. 実行

MCP サーバーを起動し、ラウンドごとに提出する。★の行が提出。

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def main() -> None:
    server = StdioServerParameters(command="uv", args=["run", "python", "server_stdio.py"], cwd=str(MCP_DIR))
    async with stdio_client(server) as (read, write), ClientSession(read, write) as session:
        await session.initialize()
        await run_agent(session)


await main()

## 6. 記録の保存

ログ・エージェントが書いたコード・提出ファイルをドライブに保存する。

In [ ]:
import shutil

from google.colab import drive

drive.mount("/content/drive")
dst = Path("/content/drive/MyDrive/uob_agent_logs") / datetime.now().strftime("%Y%m%d_%H%M%S")
dst.mkdir(parents=True, exist_ok=True)
for name in ("events.jsonl", "mcp_stderr.log"):
    if (WORK / name).exists():
        shutil.copy2(WORK / name, dst / name)
for folder in ("agent_workspace", "submissions"):
    if (WORK / folder).exists():
        shutil.copytree(WORK / folder, dst / folder, ignore=shutil.ignore_patterns("train_data", "*.zip"))
print("保存しました:", dst)